# Intro to data querying in Python: Translation Assignment

For the weHoop packages, were going to be using a package called SportsDataVerse. The goal of sportsdataverse-py is to provide the community with a python package for working with sports data as a companion to the cfbfastR, hoopR, and wehoop R packages. Beyond data aggregation and tidying ease, one of the multitude of services that sportsdataverse-py provides is for benchmarking open-source expected points and win probability metrics for American Football.

This package can be installed using:

In [16]:
# pip install sportsdataverse

Load the data

In [1]:
from sportsdataverse import wbb
import pandas as pd

wbb_2022 = wbb.load_wbb_player_boxscore(seasons=2022)

100%|██████████| 1/1 [00:00<00:00,  1.02it/s]


Convert to pandas DataFrame if not already. (I don't know what Polars is and I don't know why it wants to use that)

In [2]:
if not hasattr(wbb_2022, 'to_pandas'):
    df = wbb_2022
else:
    df = wbb_2022.to_pandas()

### Q1: During the 2022 season, what player had the most assists in a single game?

In [3]:
most_assists = df.sort_values('assists', ascending=False).head(1)
player_name = most_assists['athlete_display_name'].values[0]
assists = most_assists['assists'].values[0]

print(f"In the 2022 season, {player_name} had the most assists in a single game with {assists} assists.\n")

In the 2022 season, Ally Knights had the most assists in a single game with 20.0 assists.



### Q2: What player had the most assists in the entire 2022 season?

In [4]:
season_assists = df.groupby('athlete_display_name')['assists'].sum().reset_index()

top_assist_player = season_assists.sort_values('assists', ascending=False).head(1)

print(f"In the entire 2022 season, {top_assist_player['athlete_display_name'].values[0]} had the most total assists with {top_assist_player['assists'].values[0]} assists.")

In the entire 2022 season, Lauren Park-Lane had the most total assists with 260.0 assists.


### Q3: During the 2024 season, how many times did a player score 20 or more points in a game where their teams scored 60 or less?

In [5]:
wbb_2024 = wbb.load_wbb_player_boxscore(seasons=2024)
wbb_2024 = wbb_2024.to_pandas()


# Filter for games where:
# 1. Player scored 20+ points (points >= 20)
# 2. Team scored 60 or less (team_score <= 60)
high_scorers_low_team = wbb_2024[
    (wbb_2024['points'] >= 20) & 
    (wbb_2024['team_score'] <= 60)
]

# Count the number of occurrences
count = len(high_scorers_low_team)

print(f"In the 2024 season, there were {count} instances where a player scored 20+ points in a game where their team scored 60 or fewer points.")

100%|██████████| 1/1 [00:00<00:00,  1.67it/s]

In the 2024 season, there were 754 instances where a player scored 20+ points in a game where their team scored 60 or fewer points.


### Q4: What team had the most rebounds in the entire period from the 2021 through 2023 seasons?

In [6]:
wbb_data = wbb.load_wbb_player_boxscore(seasons=[2021, 2022, 2023])
wbb_data = wbb_data.to_pandas()

wnba_rebounds = (
    wbb_data
    .groupby(['team_id', 'team_name'])['rebounds']  # Group and select rebounds
    .sum()                                         # Sum rebounds per team
    .reset_index()                                 # Convert index to columns
    .sort_values('rebounds', ascending=False)      # Sort by rebounds (best first)
)

top_rebound_team = wnba_rebounds.iloc[0] # Get the best team

# Display results
print(f"The team with the most rebounds from 2021-2023 was {top_rebound_team['team_name']} with {top_rebound_team['rebounds']:} total rebounds.")

100%|██████████| 3/3 [00:01<00:00,  1.74it/s]


The team with the most rebounds from 2021-2023 was Gamecocks with 4588.0 total rebounds.


#### Equivalent SQL Code:
```
SELECT team_id, team_name, SUM(rebounds)  
FROM wbb_data  
GROUP BY team_id, team_name  
```

#### Equivalent R Code:

```
wnba_rebounds <- wnba |>  
  group_by(team_id) |>  
  summarise( rebounds = sum(rebounds, na.rm = TRUE), team_name) |>   
  arrange(desc(rebounds))  
  ```

### Among players with at least 50 three-point field goal attempts in a season what player has the highest single-season three point field goal percentage in the time period 2021--2025? 

In [7]:
# Load the data
data = wbb.load_wbb_player_boxscore(seasons=[2021, 2022, 2023, 2024, 2025])
df = data.to_pandas()

# Step 1: Calculate total attempts and makes for each player each season
player_stats = df.groupby(['athlete_display_name', 'season']).agg(
    attempts=('three_point_field_goals_attempted', 'sum'),
    made=('three_point_field_goals_made', 'sum')  # Note plural "goals" here too
).reset_index()

# Step 2: Only keep players with 50+ attempts
qualified = player_stats[player_stats['attempts'] >= 50]

# Step 3: Calculate percentage
qualified['percentage'] = qualified['made'] / qualified['attempts']

# Step 4: Find the best shooter
best_shooter = qualified.sort_values('percentage', ascending=False).iloc[0]

# Print the results
print("\nBest 3-Point Shooter (minimum 50 attempts, 2021-2025 seasons):")
print(f"Player: {best_shooter['athlete_display_name']}")
print(f"Season: {best_shooter['season']}")
print(f"3-Point Percentage: {best_shooter['percentage']:.1%}")
print(f"Shots: Made {int(best_shooter['made'])} out of {int(best_shooter['attempts'])}")

100%|██████████| 5/5 [00:03<00:00,  1.59it/s]



Best 3-Point Shooter (minimum 50 attempts, 2021-2025 seasons):
Player: Gianna Kneepkens
Season: 2024
3-Point Percentage: 54.0%
Shots: Made 27 out of 50


/var/folders/wf/_rnchz9x14366p91s96x6x0h0000gn/T/ipykernel_70270/1583632680.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  qualified['percentage'] = qualified['made'] / qualified['attempts']
